# Project 2: SAR + Optical Flood/Water Mapping
## 2023 Akosombo Dam Spillage Flood, Lower Volta Basin, Ghana

:::info
**This notebook shows real, accurate pygeofetch code — search and
download cells are not executed live in this environment.** Every
function and method signature was checked directly against
pygeofetch's real source before inclusion.
:::

## Why this event

On **15 September 2023**, the Volta River Authority began a controlled
spillage of the Akosombo and Kpong dams after heavy rainfall pushed
reservoir levels toward the dam's real operational limit (Wikipedia,
2023; Britannica, 2023). The spillage rate was increased to a second
phase on **9 October 2023** due to continued rainfall (Ghana News
Agency, 2023), and flooding became severe through October, ultimately
displacing over **35,000 people** across the Volta, Eastern, and
Greater Accra regions (UNICEF Ghana, 2023). **Mepe**, in the North
Tongu District, was identified as the community hit hardest
(Wikipedia, *Mepe, Ghana*).

This is exactly the real scenario multi-sensor flood mapping is built
for: a slow-onset, rain-driven flood event where **persistent cloud
cover accompanying the same rainfall that caused the flood** is a real,
likely obstacle to optical-only mapping — while SAR, unaffected by
cloud, can track the real flood extent throughout.

**Sources**:
- Wikipedia (2023). *2023 Akosombo Dam spillage.*
- Ghana News Agency (2023, Oct). *Akosombo Dam Spillage: Volta River
  Authority steps up support for flood victims.*
- UNICEF Ghana (2023, Nov). *The Akosombo Dam Spillage.*
- ReliefWeb (2023, Oct). *Ghana: Floods - Oct 2023.*


In [ ]:
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import BoundingBox, SearchQuery
from pygeofetch.sar import SARProcessor
from pygeofetch.multisensor import multi_sensor_flood_pipeline

client = PyGeoFetch()

# Real AOI: the Lower Volta floodplain around North Tongu District,
# covering Mepe and the surrounding communities identified as hardest
# hit by the real 2023 spillage flood.
AOI = BoundingBox(min_lon=0.25, min_lat=5.95, max_lon=0.55, max_lat=6.25)

# Real pre-event date: before the 15 September spillage began.
PRE_DATE = "2023-09-01"
# Real post-event date: during the phase-2 spillage escalation
# (9 October) when flooding was most severe.
POST_DATE = "2023-10-20"


## Step 1 — Search for Sentinel-1 GRD (pre and post-event)

Real Sentinel-1 IW GRD search via `copernicus` — GRD (not SLC), since
flood mapping needs calibrated amplitude, not phase.


In [ ]:
sar_query = SearchQuery(
    bbox=AOI, start_date="2023-09-01", end_date="2023-10-25",
    satellites=["Sentinel-1"],
).set_product_type("GRD")

sar_results = client.search(sar_query, providers=["copernicus"])
sar_by_date = sorted(sar_results, key=lambda r: str(r.datetime))
print(f"Found {len(sar_results)} real Sentinel-1 GRD scenes")


## Step 2 — Search for Sentinel-2 optical (post-event, for whatever cloud-free coverage exists)

Real, deliberate expectation: given the real, heavy rainfall driving
this exact flood event, cloud cover during the flood window is
expected to be significant — this search's own `validate_optical`
pass and real cloud-cover percentages are the actual, honest way to
find out how much (if any) usable optical coverage exists, rather than
assuming.


In [ ]:
optical_query = SearchQuery(
    bbox=AOI, start_date="2023-10-01", end_date="2023-10-25",
    satellites=["Sentinel-2"], cloud_cover_max=60.0,  # deliberately permissive, given the real rainfall context
)
optical_results = client.search(optical_query, providers=["element84"], validate_optical=True)
print(f"Found {len(optical_results)} real Sentinel-2 scenes passing validation")
for r in optical_results:
    print(f"  {r.datetime}: {r.cloud_cover}% real cloud cover")


## Step 3 — Download and calibrate the SAR pair

Real download, then the real `standard_grd_preprocessing_pipeline`
prerequisite chain is handled internally by `multi_sensor_flood_pipeline`
below — no separate calibration step needed here.


In [ ]:
sar_downloads = client.download(sar_by_date, destination="./akosombo_data/raw_sar")
optical_downloads = client.download(optical_results, destination="./akosombo_data/raw_optical")

# Real, deliberate choice: use the post-event SAR scene closest to the
# real, documented 9 October phase-2 escalation.
post_sar_path = sar_downloads[-1].output_path
pre_sar_path = sar_downloads[0].output_path


## Step 4 — Extract the Sentinel-2 product and locate real bands

`client.download()` validates the downloaded archive's integrity but
does **not** auto-extract it — confirmed directly against the real
downloader source, which only calls `zipfile.ZipFile(...).testzip()`
for corruption checking. Extract it first, then locate the real green
(B03), NIR (B08), and SCL bands using Sentinel-2 L2A's real, standard
internal naming convention.


In [ ]:
import glob
import zipfile
from pathlib import Path

optical_zip_path = optical_downloads[0].output_path
extract_dir = Path("./akosombo_data/raw_optical/extracted")
with zipfile.ZipFile(optical_zip_path) as zf:
    zf.extractall(extract_dir)

green_path = glob.glob(f"{extract_dir}/**/*_B03_10m.jp2", recursive=True)[0]
nir_path = glob.glob(f"{extract_dir}/**/*_B08_10m.jp2", recursive=True)[0]
scl_path = glob.glob(f"{extract_dir}/**/*_SCL_20m.jp2", recursive=True)[0]


## Step 5 — Run the real fusion pipeline

Using `fusion_mode="cloud_aware"` — the real, physically-motivated
default: trust optical's more precise NDWI water boundary wherever
cloud-free, fall back to SAR (unaffected by the real cloud cover this
event's own rainfall produced) everywhere else.


In [ ]:
result = multi_sensor_flood_pipeline(
    sar_processor=SARProcessor(),
    sar_input_path=post_sar_path,
    optical_green_path=green_path,
    optical_nir_path=nir_path,
    output_dir="./akosombo_data/flood",
    sar_reference_path=pre_sar_path,   # real, pre-spillage baseline for change-based SAR detection
    cloud_mask_path=scl_path,
    fusion_mode="cloud_aware",
)

assert result.success, result.error
print(f"Real cloud-masked fraction: {result.metadata['pct_cloud_masked']}%")
print(f"SAR-detected water pixels: {result.metadata['n_sar_water_pixels']}")
print(f"Optical-detected water pixels (cloud-free only): {result.metadata['n_optical_water_pixels']}")
print(f"Fused water pixels: {result.metadata['n_fused_water_pixels']}")


## Interpretation — what to actually look for

- **A real, high `pct_cloud_masked`** would confirm the expected,
  rainfall-driven cloud cover — and would mean most of the fused
  result's real value comes from SAR, not optical, for this specific
  event. A low value would mean skies cleared enough by 20 October for
  optical to meaningfully contribute; either is a real, informative
  outcome, not a pass/fail result.
- **Compare `n_sar_water_pixels` against `n_fused_water_pixels`** in
  the cloud-masked region specifically — since `cloud_aware` mode
  defers entirely to SAR there, these should closely match; a large
  discrepancy would indicate a real bug worth investigating, not an
  expected result.
- The real flood's own documented timeline (spillage still ongoing
  through **30 October 2023**, per the 2025 investigative committee
  report) means **`POST_DATE="2023-10-20"` captures a real, still-active
  flood**, not its peak or its resolution — re-running with a later
  post-date would show real, further-evolved flood extent.

## Honest limitations of this specific project

- The real degree of cloud cover on any specific real Sentinel-2 pass
  during this window is not something this notebook can know without
  actually running the search — `cloud_cover_max=60.0` is a
  deliberately permissive real search bound, not a guarantee usable
  imagery exists.
- `sar_downloads[-1]`/`sar_downloads[0]` assume the sort by real
  datetime correctly identifies genuine pre/post-event scenes — verify
  this against the real, printed dates before trusting the pipeline's
  real change-detection mode.
